### Boradcasting jon vs normal join

In [0]:
### Transaction (Big dataset)

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("JoinExample").getOrCreate()

txn_data = [
    (1, 101, 1001, 500, "2025-01-01"),
    (2, 102, 1002, 200, "2025-01-02"),
    (3, 103, 1003, 300, "2025-02-01"),
    (4, 101, 1004, 700, "2025-02-10"),
    (5, 104, 1005, 100, "2025-03-01"),
    (6, 105, 1006, 900, "2025-03-05")
]

txn_cols = ["txn_id", "customer_id", "product_id", "amount", "txn_date"]

txn_df = spark.createDataFrame(txn_data, txn_cols)
txn_df.display()

txn_id,customer_id,product_id,amount,txn_date
1,101,1001,500,2025-01-01
2,102,1002,200,2025-01-02
3,103,1003,300,2025-02-01
4,101,1004,700,2025-02-10
5,104,1005,100,2025-03-01
6,105,1006,900,2025-03-05


In [0]:
## customer small dataset

cust_data = [
    (101, "Arjun", "Mumbai", "Gold"),
    (102, "Ravi", "Delhi", "Silver"),
    (103, "Priya", "Bangalore", "Gold"),
    (104, "Neha", "Chennai", "Bronze"),
    (105, "Amit", "Pune", "Silver")
]

cust_cols = ["customer_id", "name", "city", "segment"]

cust_df = spark.createDataFrame(cust_data, cust_cols)
cust_df.display()

customer_id,name,city,segment
101,Arjun,Mumbai,Gold
102,Ravi,Delhi,Silver
103,Priya,Bangalore,Gold
104,Neha,Chennai,Bronze
105,Amit,Pune,Silver


In [0]:
print("Txn Count:", txn_df.count())
print("Cust Count:", cust_df.count())

Txn Count: 6
Cust Count: 5


In [0]:
from pyspark.sql.functions import broadcast

broadcast_df = txn_df.join(
    broadcast(cust_df),
    on="customer_id",
    how="inner"
)

broadcast_df.show()

+-----------+------+----------+------+----------+-----+---------+-------+
|customer_id|txn_id|product_id|amount|  txn_date| name|     city|segment|
+-----------+------+----------+------+----------+-----+---------+-------+
|        101|     1|      1001|   500|2025-01-01|Arjun|   Mumbai|   Gold|
|        102|     2|      1002|   200|2025-01-02| Ravi|    Delhi| Silver|
|        103|     3|      1003|   300|2025-02-01|Priya|Bangalore|   Gold|
|        101|     4|      1004|   700|2025-02-10|Arjun|   Mumbai|   Gold|
|        104|     5|      1005|   100|2025-03-01| Neha|  Chennai| Bronze|
|        105|     6|      1006|   900|2025-03-05| Amit|     Pune| Silver|
+-----------+------+----------+------+----------+-----+---------+-------+



In [0]:
broadcast_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [customer_id])
:- 'UnresolvedSubqueryColumnAliases [txn_id, customer_id, product_id, amount, txn_date]
:  +- LocalRelation [txn_id#10970L, customer_id#10971L, product_id#10972L, amount#10973L, txn_date#10974]
+- 'UnresolvedHint broadcast
   +- 'UnresolvedSubqueryColumnAliases [customer_id, name, city, segment]
      +- LocalRelation [customer_id#11007L, name#11008, city#11009, segment#11010]

== Analyzed Logical Plan ==
customer_id: bigint, txn_id: bigint, product_id: bigint, amount: bigint, txn_date: string, name: string, city: string, segment: string
Project [customer_id#11120L, txn_id#11119L, product_id#11121L, amount#11122L, txn_date#11123, name#11125, city#11126, segment#11127]
+- Join Inner, (customer_id#11120L = customer_id#11124L)
   :- Project [txn_id#10970L AS txn_id#11119L, customer_id#10971L AS customer_id#11120L, product_id#10972L AS product_id#11121L, amount#10973L AS amount#11122L, txn_date#10974 AS txn_date#11123]
   :  +

In [0]:
###  Normal Join (Optimized Flow

txn_filtered = txn_df.filter("txn_date >= '2025-02-01'")  # filter

txn_filtered = txn_filtered.select("customer_id", "amount") # select required column
cust_filtered = cust_df.select("customer_id", "segment") # select required column

txn_repart = txn_filtered.repartition(4, "customer_id")
cust_repart = cust_filtered.repartition(4, "customer_id")



In [0]:
txn_filtered.display()
cust_filtered.display()

customer_id,amount
103,300
101,700
104,100
105,900


customer_id,segment
101,Gold
102,Silver
103,Gold
104,Bronze
105,Silver


In [0]:
txn_repart.display()
cust_repart.display()

customer_id,amount
101,700
104,100
103,300
105,900


customer_id,segment
101,Gold
102,Silver
104,Bronze
103,Gold
105,Silver


In [0]:
normal_df = txn_repart.join(
    cust_repart,
    on="customer_id",
    how="inner"
)

normal_df.show()

+-----------+------+-------+
|customer_id|amount|segment|
+-----------+------+-------+
|        101|   700|   Gold|
|        104|   100| Bronze|
|        103|   300|   Gold|
|        105|   900| Silver|
+-----------+------+-------+



In [0]:
# Enable AQE (recommended)
# spark.conf.set("spark.sql.adaptive.enabled", "true")

# Check default shuffle partitions
spark.conf.get("spark.sql.shuffle.partitions")

'auto'

In [0]:
from pyspark.sql.functions import broadcast

# Step 1: Filter
txn_filtered = txn_df.filter("txn_date >= '2025-02-01'")

# Step 2: Select
txn_filtered = txn_filtered.select("customer_id", "amount")
cust_filtered = cust_df.select("customer_id", "segment")

# Step 3: Decide Join Type
if cust_filtered.count() < 1000:   # example condition
    final_df = txn_filtered.join(
        broadcast(cust_filtered),
        "customer_id"
    )
else:
    txn_repart = txn_filtered.repartition(4, "customer_id")
    cust_repart = cust_filtered.repartition(4, "customer_id")

    final_df = txn_repart.join(
        cust_repart,
        "customer_id"
    )

# Step 4: Output
final_df.show()

+-----------+------+-------+
|customer_id|amount|segment|
+-----------+------+-------+
|        103|   300|   Gold|
|        101|   700|   Gold|
|        104|   100| Bronze|
|        105|   900| Silver|
+-----------+------+-------+



### Use Cache

In [0]:
data = [
    (1, "Alice", "India", 1000),
    (2, "Bob", "USA", 2000),
    (3, "Charlie", "India", 1500),
    (4, "David", "USA", 3000),
    (5, "Eve", "India", 2500),
    (2, "Edeh", "India", 6500)
]

columns = ["id", "name", "country", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

id,name,country,sales
1,Alice,India,1000
2,Bob,USA,2000
3,Charlie,India,1500
4,David,USA,3000
5,Eve,India,2500
2,Edeh,India,6500


In [0]:
from pyspark.sql.functions import sum,collect_list

In [0]:
df_sale = df.groupBy("id").agg(collect_list("sales").alias("sale"))
df_sale.display(Truncate=True)

id,sale
1,List(1000)
2,"List(2000, 6500)"
3,List(1500)
4,List(3000)
5,List(2500)


In [0]:
## Transformation and First ACtion
df_transformed = df.groupBy("country").agg(sum("sales").alias("Total_Sales"))
df_transformed.display()

country,Total_Sales
India,11500
USA,5000


In [0]:
# First usage
df_transformed.filter("country = 'India'").show()

# Second usage
df_transformed.orderBy("Total_Sales", ascending=False).show()

+-------+-----------+
|country|Total_Sales|
+-------+-----------+
|  India|       5000|
+-------+-----------+

+-------+-----------+
|country|Total_Sales|
+-------+-----------+
|  India|       5000|
|    USA|       5000|
+-------+-----------+



### handle skwed data

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Skewed data: A has huge data, others small
data = [
    (101, "order1", 100),
    (101, "order2", 200),
    (101, "order3", 300),
    (101, "order4", 400),
    (102, "order5", 150),
    (103, "order6", 200)
]

columns = ["customer_id", "order_id", "amount"]

orders_df = spark.createDataFrame(data, columns)
orders_df.display()

customer_id,order_id,amount
101,order1,100
101,order2,200
101,order3,300
101,order4,400
102,order5,150
103,order6,200


In [0]:
customer_data = [
    (101, "Argha"),
    (102, "Rahul"),
    (103, "Amit")
]

customer_df = spark.createDataFrame(customer_data, ["customer_id", "customer_name"])
customer_df.display()

customer_id,customer_name
101,Argha
102,Rahul
103,Amit


In [0]:
from pyspark.sql.functions import floor, rand
from pyspark.sql.functions import explode, array, lit, col

In [0]:
salt_range = 3

orders_salted = orders_df.withColumn(
    "salt", floor(rand() * salt_range)
)

orders_salted.display()

customer_id,order_id,amount,salt
101,order1,100,0
101,order2,200,1
101,order3,300,1
101,order4,400,1
102,order5,150,0
103,order6,200,2


In [0]:
customer_expanded = customer_df.withColumn(
    "salt",
    explode(array([lit(i) for i in range(salt_range)]))
)

customer_expanded.display()

customer_id,customer_name,salt
101,Argha,0
101,Argha,1
101,Argha,2
102,Rahul,0
102,Rahul,1
102,Rahul,2
103,Amit,0
103,Amit,1
103,Amit,2


In [0]:
salted_join = orders_salted.join(
    customer_expanded,
    ["customer_id", "salt"]
)

salted_join.display()

customer_id,salt,order_id,amount,customer_name
101,1,order1,100,Argha
101,0,order2,200,Argha
101,0,order3,300,Argha
101,2,order4,400,Argha
102,0,order5,150,Rahul
103,2,order6,200,Amit


In [0]:
final_df = salted_join.drop("salt")
final_df.display()

customer_id,order_id,amount,customer_name
101,order1,100,Argha
101,order2,200,Argha
101,order3,300,Argha
101,order4,400,Argha
102,order5,150,Rahul
103,order6,200,Amit


### Remove Duplicate

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

data = [
    (101, "Argha", "Kolkata", "2026-03-28 10:00:00"),
    (101, "Argha", "Mumbai",  "2026-03-28 12:00:00"),
    (102, "Rahul", "Delhi",   "2026-03-28 09:30:00"),
    (102, "Rahul", "Delhi",   "2026-03-28 09:30:00"),
    (103, "Sneha", "Pune",    "2026-03-28 11:15:00"),
    (103, "Sneha", "Bangalore","2026-03-28 11:45:00")
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("updated_at", StringType(), True)
])

df = spark.createDataFrame(data, schema)

df = df.withColumn("updated_at", to_timestamp(col("updated_at")))

df.display(truncate=False)

customer_id,customer_name,city,updated_at
101,Argha,Kolkata,2026-03-28T10:00:00.000Z
101,Argha,Mumbai,2026-03-28T12:00:00.000Z
102,Rahul,Delhi,2026-03-28T09:30:00.000Z
102,Rahul,Delhi,2026-03-28T09:30:00.000Z
103,Sneha,Pune,2026-03-28T11:15:00.000Z
103,Sneha,Bangalore,2026-03-28T11:45:00.000Z


In [0]:
window_spec = Window.partitionBy(col("customer_id")).orderBy(col("updated_at").desc())
source_dedup = df.withColumn("rn",row_number().over(window_spec)).filter(col('rn')==1)
source_dedup.display()

customer_id,customer_name,city,updated_at,rn
101,Argha,Mumbai,2026-03-28T12:00:00.000Z,1
102,Rahul,Delhi,2026-03-28T09:30:00.000Z,1
103,Sneha,Bangalore,2026-03-28T11:45:00.000Z,1


In [0]:
df_no_duplicates = df.dropDuplicates()
df_no_duplicates.display()

customer_id,customer_name,city,updated_at
101,Argha,Kolkata,2026-03-28T10:00:00.000Z
101,Argha,Mumbai,2026-03-28T12:00:00.000Z
102,Rahul,Delhi,2026-03-28T09:30:00.000Z
103,Sneha,Pune,2026-03-28T11:15:00.000Z
103,Sneha,Bangalore,2026-03-28T11:45:00.000Z


In [0]:
from pyspark.sql.functions import col, explode
data = [{
  "id": 1,
  "name": "Argha",
  "address": {
    "city": "Mumbai",
    "state": "MH"
  },
  "orders": [
    {
      "order_id": 101,
      "amount": 250
    },
    {
      "order_id": 102,
      "amount": 450
    }
  ]
}]
df = spark.createDataFrame(data)

In [0]:
df.printSchema()
df.display()

root
 |-- address: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)



address,id,name,orders
"Map(city -> Mumbai, state -> MH)",1,Argha,"List(Map(order_id -> 101, amount -> 250), Map(order_id -> 102, amount -> 450))"


In [0]:
df1 = df.select(
    col("id"),col("name"),col("address.city").alias("city"),col("address.state").alias("state"),col("orders")
)
df2 = df1.withColumn("order",explode(col("orders")))
df2 = df2.select(col("id"),col("name"),col("city"),col("state"),\
                col('order.order_id').alias("order_id"),col('order.amount').alias("amount") )
df2.display()

id,name,city,state,order_id,amount
1,Argha,Mumbai,MH,101,250
1,Argha,Mumbai,MH,102,450
